<a href="https://colab.research.google.com/github/kokila83/agenticai/blob/main/ex_3_agent_with_multiple_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import getpass
import json
from datetime import datetime
from openai import OpenAI

# 1. Get Azure AI Foundry API key securely
api_key = getpass.getpass("Enter your Azure AI Foundry API key: ")

# 2. Configure OpenAI client pointing to your Azure AI Foundry endpoint
client = OpenAI(
    api_key=api_key.strip(),
    base_url="https://ganapathy-0333-resource.services.ai.azure.com/openai/v1/"
)

# 3. Define available tools
tools = {
    "get_time": lambda: datetime.now().strftime("%Y-%m-%d %H:%M"),
    "calculate": lambda expr: str(eval(expr)),
    "get_weather": lambda city: f"Weather in {city}: 25°C, Sunny"
}

# 4. Agent runner function
def agent_run(query):
    prompt = f"""You are an AI agent with these tools:
- get_time(): Returns current date and time
- calculate(expr): Evaluates math
- get_weather(city): Returns weather

User query: {query}

Respond ONLY with valid JSON in one of these two formats:
{{"tool": "name", "args": ["arg1"]}}
OR
{{"tool": "none", "response": "direct answer"}}"""

    try:
        # Call your deployed Azure AI Foundry model
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[{"role": "user", "content": prompt}]
        )

        raw_output = response.choices[0].message.content.strip()

        # Parse JSON decision from the model
        decision = json.loads(raw_output)

        if decision.get("tool") and decision["tool"] != "none":
            tool_name = decision["tool"]
            args = decision.get("args", [])

            if tool_name in tools:
                tool_result = tools[tool_name](*args)
                print(f"Agent used '{tool_name}' tool with args {args} -> {tool_result}")
                return tool_result
            else:
                return f"Error: Tool '{tool_name}' not found."

        return decision.get("response", raw_output)

    except json.JSONDecodeError:
        return raw_output
    except Exception as e:
        return f"Execution Error: {e}"

# 5. Run test queries
if __name__ == "__main__":
    print(agent_run("What time is it?"))
    print("-" * 50)
    print(agent_run("What's 25 * 48?"))

Enter your Azure AI Foundry API key: ··········
Agent used 'get_time' tool with args [] -> 2026-09-27 00:23
2026-09-27 00:23
--------------------------------------------------
Agent used 'calculate' tool with args ['25 * 48'] -> 1200
1200
